## Opening Dataset from preprocessing and loading libraries

In [ ]:
from bertopic import BERTopic
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np  
import umap
import hdbscan
from sentence_transformers import SentenceTransformer
from google import genai
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance, OpenAI, PartOfSpeech
import spacy 
from umap import UMAP
from hdbscan import HDBSCAN
# Load the data
df_filtered = pd.read_csv("C:\\Users\\20193623\\OneDrive - TU Eindhoven\\BEP\\Mijn project\\data\\cleaned\\df_filtered.csv")
df_positive = pd.read_csv("C:\\Users\\20193623\\OneDrive - TU Eindhoven\\BEP\\Mijn project\\data\\cleaned\\df_positive.csv")
df_negative = pd.read_csv("C:\\Users\\20193623\\OneDrive - TU Eindhoven\\BEP\\Mijn project\\data\\cleaned\\df_negative.csv")

#Dit werkte dus niet 
# docs = df_filtered['merged_lies'].tolist()
# docs_positive = df_positive['merged_lies'].tolist()
# docs_negative = df_negative['merged_lies'].tolist()

BELANGRIJK STAP VOOR HET OMZETTEN NAAR DOCS -> De merged lies column werkte niet want je wil juist alle losse leugens als input en niet de som van totale input want dan worden de verschillende topics in de leugens als samenhangend gezien en dat is niet


In [36]:
import pandas as pd

# Define a function to extract all individual lies from a dataframe.
def extract_lies(dataframe):
    """Gathers all individual lies from the Embedded_lies_* columns."""
    
    # Create a list of all possible column names, from 'Embedded_lies_1' to 'Embedded_lies_20'.
    lie_columns = [f'Embedded_lies_{i}' for i in range(1, 21)]
    docs = []
    
    for index, row in dataframe.iterrows():
        for col in lie_columns:
            if col in dataframe.columns:
                lie_text = row[col]
                if pd.notna(lie_text) and str(lie_text).strip() != '':
                    docs.append(str(lie_text))
    return docs


docs_positive = extract_lies(df_positive)
docs_negative = extract_lies(df_negative)
docs = extract_lies(df_filtered)


# Print the number of documents in each list to verify the process worked as expected.
# This is a good sanity check to see if the counts make sense.
print(f"Number of individual 'Positive Gain' lies: {len(docs_positive)}")
print(f"Number of individual 'Negative Gain' lies: {len(docs_negative)}")
print(f"Total number of individual lies: {len(docs_all)}")

Number of individual 'Positive Gain' lies: 1528
Number of individual 'Negative Gain' lies: 1728
Total number of individual lies: 3256


# Start embedding and saving them for optimization


In [37]:
embedding_model = SentenceTransformer("all-mpnet-base-v2")
embeddings = embedding_model.encode(docs, show_progress_bar=True)

Batches: 100%|██████████| 102/102 [00:47<00:00,  2.13it/s]


In [49]:
umap_model = UMAP(n_neighbors=30, n_components=5, min_dist=0.0, metric='cosine', random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size=30, metric='euclidean', cluster_selection_method='eom', prediction_data=True)
vectorizer_model = CountVectorizer(stop_words="english", min_df=1, ngram_range=(1, 2))


In [50]:

# KeyBERT
keybert_model = KeyBERTInspired()

# Part-of-Speech
pos_model = PartOfSpeech("en_core_web_sm")

# MMR
mmr_model = MaximalMarginalRelevance(diversity=0.3)

# All representation models
representation_model = {
    "KeyBERT": keybert_model,
    "MMR": mmr_model,
    "POS": pos_model
}

In [51]:
from bertopic import BERTopic

topic_model = BERTopic(

  # Pipeline models
  embedding_model=embedding_model,
  umap_model=umap_model,
  hdbscan_model=hdbscan_model,
  vectorizer_model=vectorizer_model,
  representation_model=representation_model,

  # Hyperparameters
  top_n_words=10,
  verbose=True
)

# Train model
topics, probs = topic_model.fit_transform(docs, embeddings)

# Show topics
topic_model.get_topic_info()
topic_model.visualize_documents(docs)

2025-09-30 12:27:35,467 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-09-30 12:27:54,548 - BERTopic - Dimensionality - Completed ✓
2025-09-30 12:27:54,548 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-09-30 12:27:54,678 - BERTopic - Cluster - Completed ✓
2025-09-30 12:27:54,678 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-09-30 12:28:10,246 - BERTopic - Representation - Completed ✓


In [54]:
topic_model.get_topic_info()


,Topic,Count,Name,Representation,KeyBERT,MMR,POS,Representative_Docs
0,-1,1320,-1_work_time_did_just,"[work, time, did, just, team, went, project, m...","[work, school, working, job, police, friends, ...","[work, time, project, day, didn, school, frien...","[time, team, project, money, day, experience, ...","[I get to work the same team, It was late at n..."
1,0,279,0_car_driving_speeding_road,"[car, driving, speeding, road, drive, driver, ...","[driving car, driving, car, drove, vehicle, dr...","[driving, speeding, road, speeding ticket, veh...","[car, driving, road, driver, accident, speed, ...","[i wasn't driving the car, either car, his car]"
2,1,275,1_surgery_pain_hospital_doctor,"[surgery, pain, hospital, doctor, infection, d...","[surgery, pain, surgical, hospital, anesthesia...","[surgery, pain, doctor, infection, room, surge...","[surgery, pain, hospital, doctor, infection, d...","[I was in so much pain, surgery, surgery]"
3,2,257,2_ticket_bus_train_inspector,"[ticket, bus, train, inspector, buy, fine, tic...","[ticket told, valid ticket, ticket card, ticke...","[ticket, bus, buy ticket, ticket inspector, pa...","[ticket, bus, train, inspector, fine, tickets,...",[When the ticket inspector came up to me after...
4,3,255,3_interview_job_got_position,"[interview, job, got, position, got job, compa...","[interview went, interview started, interview ...","[interview, got job, interviewer, interview we...","[interview, job, position, company, interviewe...","[and the interview went very well, The positio..."
5,4,228,4_nervous_life_felt_confident,"[nervous, life, felt, confident, did, happy, r...","[nervous, nervous fully, nervous felt, nervous...","[nervous, felt, confident, happy, excited, cal...","[nervous, life, confident, happy, excited, con...","[very nervous, I got even more nervous because..."
6,5,137,5_exam_questions_test_studied,"[exam, questions, test, studied, passed, cheat...","[pass exam, exam, exam day, prepared exam, exa...","[exam, test, studied, passed, cheat, study, qu...","[exam, questions, test, paper, notes, answers,...","[I was not prepared to the exam, cheating duri..."
7,6,123,6_months_ago_late_minutes,"[months, ago, late, minutes, time, hours, hurr...","[months ago, years ago, ago, earlier, months, ...","[late, minutes, time, hurry, time years, month...","[months, late, minutes, time, hours, hurry, mo...","[Four months ago, Four months ago, 6 months ago]"
8,7,108,7_company_town_events_property,"[company, town, events, property, mathematics,...","[company, business, partnerships, partnerships...","[company, events, networking events, attending...","[company, town, events, property, sales, offic...","[company, our company, company]"
9,8,76,8_boss_manager_supervisor_want,"[boss, manager, supervisor, want, said, collea...","[blame boss, boss, told bosses, new boss, fire...","[boss, supervisor, leave, new boss, warranty c...","[boss, manager, supervisor, colleague, office,...","[blame my boss, They both said to if I want th..."
